# Rekaman Prompt Hari 2 — EDA Lapis 2–4 (Langkah 4–6)

Pelatihan Machine Learning untuk Operasi PLTP · Hari 2, 10.40–11.30 · deck `materi-3s-praktek-eda`

> **Rencana D — rekaman prompt.** Notebook ini dipakai bila AI coding agent tidak bisa
> diakses dari tempat pelatihan. Setiap langkah berisi *prompt* (sama dengan di slide) dan
> *kode yang dihasilkan AI* untuk prompt itu. Tugas Anda tetap sama dengan vibe coding:
> **baca kodenya, tebak hasilnya, jalankan, lalu periksa.**
>
> Begitu akses AI tersedia (VPN, hotspot, OpenCode, atau Google Colab), ketik prompt yang sama
> ke agent dan bandingkan kodenya dengan rekaman ini. Tidak harus sama persis — yang harus
> sama adalah *angkanya*.

### Persiapan — data bersih

> **Prompt:** Baca `data/produksi_bulanan.csv`, seragamkan `kode_unit` ke huruf besar, hapus duplikat, dan ubah `tanggal` menjadi tipe tanggal (sama dengan Langkah 2). Cetak jumlah baris dan unit.

**Tebak dulu sebelum menjalankan sel di bawah:** berapa baris dan berapa unit?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/produksi_bulanan.csv")
df["kode_unit"] = df["kode_unit"].str.upper()
df = df.drop_duplicates().reset_index(drop=True)
df["tanggal"] = pd.to_datetime(df["tanggal"])

gangguan = pd.read_csv("../data/gangguan.csv")
unit = pd.read_csv("../data/unit_pembangkit.csv")
print("Data bersih:", len(df), "baris,", df["kode_unit"].nunique(), "unit")

## Lapis 2 — Satu variabel

### Langkah 4a — sebaran CF

> **Prompt:** Pada data yang sudah dibersihkan: buat histogram `cf_persen`, lalu tabel rata-rata, median, minimum, dan n per unit.

**Tebak dulu sebelum menjalankan sel di bawah:** rata-rata atau median yang lebih tinggi? Kenapa?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
df["cf_persen"].plot(kind="hist", bins=range(0, 101, 5), ax=ax, edgecolor="white")
ax.set_title("Sebaran CF bulanan — 1.080 unit-bulan")
ax.set_xlabel("CF (%)")
ax.set_ylabel("jumlah unit-bulan")
plt.show()

print("Rata-rata CF:", round(df["cf_persen"].mean(), 1), "| median:", round(df["cf_persen"].median(), 1))
tabel = df.groupby("kode_unit")["cf_persen"].agg(rata_rata="mean", median="median", minimum="min", n="count")
display(tabel.round(1))

**Periksa sebelum percaya:**

- [ ] Rata-rata 83,4 % lebih rendah dari median 85,3 % — ekor panjang ke kiri.
- [ ] Setiap unit n = 180. Kalau ada unit dengan n 1 atau 178, data belum bersih.
- [ ] Minimum per unit 2,6–19,6 % — salah data, atau unit berhenti?

### Langkah 4b — outlier, lalu cocokkan dengan gangguan

> **Prompt:** Hitung batas outlier `cf_persen` dengan aturan IQR. Daftar unit-bulan di luar batas, lalu cocokkan dengan `data/gangguan.csv` (unit dan bulan yang sama). Berapa yang punya gangguan, dan jenis apa?

**Tebak dulu sebelum menjalankan sel di bawah:** berapa outlier yang punya catatan gangguan?

In [ ]:
q1, q3 = df["cf_persen"].quantile([0.25, 0.75])
iqr = q3 - q1
bawah, atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Batas IQR: bawah {bawah:.1f} %, atas {atas:.1f} %")
print("Di bawah batas:", (df["cf_persen"] < bawah).sum(), "| di atas batas:", (df["cf_persen"] > atas).sum())

outlier = df[(df["cf_persen"] < bawah) | (df["cf_persen"] > atas)].copy()
outlier["bulan_ini"] = outlier["tanggal"].dt.to_period("M")
gangguan["bulan_mulai"] = pd.to_datetime(gangguan["tanggal_mulai"]).dt.to_period("M")

# Untuk setiap outlier: jenis gangguan apa yang dimulai pada unit dan bulan yang sama?
cocok = outlier.merge(gangguan[["kode_unit", "bulan_mulai", "jenis"]],
                      left_on=["kode_unit", "bulan_ini"], right_on=["kode_unit", "bulan_mulai"], how="left")
jenis = (cocok.groupby(["kode_unit", "bulan_ini"])["jenis"]
              .apply(lambda s: ",".join(sorted(s.dropna().unique())) or "tidak ada"))
print("Outlier yang punya gangguan:", (jenis != "tidak ada").sum(), "dari", len(jenis))
print(jenis.value_counts())

# Yang paling ekstrem: CF di bawah 50 %
rendah = outlier[outlier["cf_persen"] < 50]
print("\nUnit-bulan dengan CF < 50 %:", len(rendah))
print(jenis.loc[list(zip(rendah["kode_unit"], rendah["bulan_ini"]))].value_counts())

**Periksa sebelum percaya:**

- [ ] Batas bawah 75,7 % → 91 unit-bulan di bawahnya, 0 di atas.
- [ ] 91 dari 91 punya gangguan di unit dan bulan yang sama.
- [ ] 17 bulan dengan CF < 50 % — semuanya MO (pemeliharaan terencana).
- [ ] Jadi outlier ini *kejadian nyata*: jangan dibuang untuk P2 (prediksi FO).

## Lapis 3 — Hubungan & kelompok

### Langkah 5a — CF per jenis turbin

> **Prompt:** Gabungkan dengan `data/unit_pembangkit.csv`. Cetak jumlah baris sebelum dan sesudah. Bandingkan CF per `jenis_turbin` dalam tabel dengan kolom n, rata-rata, dan median. Apa lagi yang ikut berbeda antar jenis turbin?

**Tebak dulu sebelum menjalankan sel di bawah:** jumlah baris berubah setelah gabung? Turbin mana CF-nya tertinggi?

In [ ]:
print("Sebelum gabung:", len(df))
gabung = df.merge(unit, on="kode_unit", how="left")
print("Sesudah gabung:", len(gabung))

per_turbin = gabung.groupby("jenis_turbin").agg(
    n=("cf_persen", "size"),
    rata_rata=("cf_persen", "mean"),
    median=("cf_persen", "median"),
    jumlah_unit=("kode_unit", "nunique"),
    kapasitas_min_mw=("kapasitas_mw", "min"),
    kapasitas_maks_mw=("kapasitas_mw", "max"),
)
display(per_turbin.round(1))

**Periksa sebelum percaya:**

- [ ] Tetap 1.080 baris setelah gabung.
- [ ] Binary 83,0 (n 540) · Single Flash 83,6 (n 360) · Double Flash 84,5 (n 180 — satu unit saja).
- [ ] Unit Binary juga yang terkecil (20–45 MW): jenis dan ukuran ikut berbeda — belum bisa disimpulkan bahwa turbin Binary lebih buruk.

### Langkah 5b — korelasi

> **Prompt:** Hitung korelasi antara `uap_aktual_ton`, `energi_aktual_mwh`, `jam_operasi`, dan `cf_persen`. Jelaskan pasangan yang korelasinya mendekati 1.

**Tebak dulu sebelum menjalankan sel di bawah:** pasangan mana yang paling mendekati 1?

In [ ]:
kolom = ["uap_aktual_ton", "energi_aktual_mwh", "jam_operasi", "cf_persen"]
display(df[kolom].corr().round(2))

**Periksa sebelum percaya:**

- [ ] uap–energi 0,99: uap menjadi listrik — dua kolom membawa informasi yang sama.
- [ ] jam operasi–CF 0,92: CF dihitung dari jam jalan.
- [ ] Korelasi tinggi di sini bukan penemuan, dan bukan sebab-akibat.

## Lapis 4 — Waktu

### Langkah 6a — rata-rata vs median per bulan

> **Prompt:** Buat grafik garis CF per bulan kalender (Jan–Des, semua tahun digabung): satu garis rata-rata dan satu garis median. Bila keduanya berbeda jauh di bulan tertentu, tampilkan unit-bulan yang menyebabkannya.

**Tebak dulu sebelum menjalankan sel di bawah:** di bulan Maret, apakah median juga turun?

In [ ]:
df["bulan_kalender"] = df["tanggal"].dt.month
per_bulan = df.groupby("bulan_kalender")["cf_persen"].agg(rata_rata="mean", median="median")

fig, ax = plt.subplots(figsize=(9, 4))
per_bulan.plot(marker="o", ax=ax)
ax.set_xticks(range(1, 13))
ax.set_xlabel("bulan")
ax.set_ylabel("CF (%)")
ax.set_title("CF per bulan kalender: rata-rata vs median")
plt.show()
display(per_bulan.round(1).T)

# Unit-bulan yang menarik rata-rata Maret dan September turun
penyebab = df[df["bulan_kalender"].isin([3, 9]) & (df["cf_persen"] < 50)]
display(penyebab[["tanggal", "kode_unit", "cf_persen", "jam_operasi"]].sort_values("tanggal"))

**Periksa sebelum percaya:**

- [ ] Rata-rata: Maret 80,0, September 80,4, Desember 86,5.
- [ ] Median hampir datar: 84,5–86,8 — bulan 'biasa' tidak berubah.
- [ ] Penurunan rata-rata datang dari beberapa unit-bulan yang anjlok.

### Langkah 6b — MO per bulan

> **Prompt:** Dari `gangguan.csv`, hitung jumlah kejadian MO per bulan mulai.

**Tebak dulu sebelum menjalankan sel di bawah:** di bulan apa MO paling sering dimulai?

In [ ]:
mo = gangguan[gangguan["jenis"] == "MO"]
print(pd.to_datetime(mo["tanggal_mulai"]).dt.month.value_counts().sort_index().rename("jumlah_MO"))

**Periksa sebelum percaya:**

- [ ] MO dimulai Maret 7, April 4, September 5, Februari 1.
- [ ] Pola jendela pemeliharaan ini bahan clustering P1 sore ini.
- [ ] Tulis temuan Lapis 2–4 sebagai kalimat (dengan angka) di prompt log — Langkah 8.